# Job ETL da silver

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. No caso desse projeto, a fonte será extraida da camada raw para a silver pelo arquivo Complete_Pokedex_V1.1.csv e o resultado será armazenado no banco e utilizado na camada gold.

### Frameworks utilizados

In [9]:
import pandas as pd
import psycopg2
import time

### EXTRACT (Extrair)


In [10]:
df = pd.read_csv('../Data_Layer/raw/Complete_Pokedex_V1.1.csv')

print("\n Extract concluído!")


 Extract concluído!


### TRANSFORM (Transformar)


Apagando colunas

In [ ]:
# Apaga colunas
colunas_para_apagar = [
'ability_1',
'ability_2',
'ability_3',
'number_pokemon_with_typing',
'primary_color',
'mean', 
'standard_deviation', 
'exp_to_level_100', 
'can_evolve', 
'final_evolution', 
'is_default', 
'baby_pokemon', 
'genus', 
'egg_group_1', 
'egg_group_2', 
'shape', 
'bmi', 
'special_attack', 
'special_defense', 
'speed',
'type_2'
]

df_tratado = df.drop(columns=colunas_para_apagar)

Removendo tuplas duplicadas

In [13]:
df_tratado = df_tratado.drop_duplicates(subset=['pokedex_number'])

Tratando nulos

In [ ]:
df_tratado[["type_2", "evolves_from"]] = df_tratado[["type_2", "evolves_from"]].fillna("N/A")
df_tratado.isnull().sum()

pokedex_number      0
pokemon_name        0
type_1              0
type_2              0
height              0
weight              0
hit_points          0
attack              0
defense             0
total_stats         0
capture_rate        0
generation          0
base_happiness      0
base_experience     0
exp_type            0
evolves_from        0
mega_evolution      0
alolan_form         0
galarian_form       0
forms_switchable    0
legendary           0
mythical            0
genderless          0
female_rate         0
egg_cycles          0
against_normal      0
against_fire        0
against_water       0
against_electric    0
against_grass       0
against_ice         0
against_fighting    0
against_poison      0
against_ground      0
against_flying      0
against_psychic     0
against_bug         0
against_rock        0
against_ghost       0
against_dragon      0
against_dark        0
against_steel       0
against_fairy       0
dtype: int64

Confererindo resultado

In [16]:
print(df_tratado.head())

print("\n Transform concluído!")

    pokedex_number   pokemon_name type_1  type_2  height  weight  hit_points  \
1                2        Ivysaur  Grass  Poison     1.0    13.0          60   
2                3  Mega Venusaur  Grass  Poison     2.4   155.5          80   
7                6      Charizard   Fire  Flying     1.7    90.5          78   
18              12     Butterfree    Bug  Flying     1.1    32.0          60   
21              14         Kakuna    Bug  Poison     0.6    10.0          45   

    attack  defense  total_stats  ...  against_ground  against_flying  \
1       62       63          405  ...             1.0             2.0   
2      100      123          625  ...             1.0             2.0   
7       84       78          534  ...             0.0             1.0   
18      45       50          395  ...             0.0             2.0   
21      25       50          205  ...             1.0             2.0   

    against_psychic  against_bug against_rock against_ghost  against_dragon  \
1

### LOAD (Carregar) 

Conectando com o banco

In [41]:
# Conecta no PostgreSQL
while True:
    try:
        conexao = psycopg2.connect(
            host="localhost", # ou conectar com o postgres quando for automático no compose
            port=5432,
            database="pokedex_db",
            user="pokedex_user",
            password="pokedex_password"
        )
        break
    except psycopg2.OperationalError:
        print("O banco não está pronto, aguardando 3 segundos...")
        time.sleep(3)

# cria o cursor
cursor = conexao.cursor()

print("Conexão efetuada com sucesso...")

Conexão efetuada com sucesso...


Criando schema silver e tabelas

In [18]:
# cria um schema para a silver
cursor.execute("""
CREATE SCHEMA IF NOT EXISTS slvr;
""")

# cria a tabela 
cursor.execute("""
CREATE TABLE IF NOT EXISTS slvr.pokemon (
    pokedex_number INT NOT NULL PRIMARY KEY,
    pokemon_name VARCHAR(50) NOT NULL,
    type_1 VARCHAR(50) NOT NULL,
    type_2 VARCHAR(50),
    height DOUBLE PRECISION NOT NULL,
    weight DOUBLE PRECISION NOT NULL,
    hit_points INT NOT NULL,
    attack INT NOT NULL,
    defense INT NOT NULL,
    total_stats INT NOT NULL,
    capture_rate INT NOT NULL,
    generation INT NOT NULL,
    base_happiness INT NOT NULL,
    base_experience INT NOT NULL,
    exp_type VARCHAR(50) NOT NULL,
    evolves_from VARCHAR(50),
    mega_evolution BOOLEAN NOT NULL,
    alolan_form BOOLEAN NOT NULL,
    galarian_form BOOLEAN NOT NULL,
    forms_switchable BOOLEAN NOT NULL,
    legendary BOOLEAN NOT NULL,
    mythical BOOLEAN NOT NULL,
    genderless BOOLEAN NOT NULL,
    female_rate DOUBLE PRECISION NOT NULL,
    egg_cycles INT NOT NULL,
    against_normal DOUBLE PRECISION NOT NULL,
    against_fire DOUBLE PRECISION NOT NULL,
    against_water DOUBLE PRECISION NOT NULL,
    against_electric DOUBLE PRECISION NOT NULL,
    against_grass DOUBLE PRECISION NOT NULL,
    against_ice DOUBLE PRECISION NOT NULL,
    against_fighting DOUBLE PRECISION NOT NULL,
    against_poison DOUBLE PRECISION NOT NULL,
    against_ground DOUBLE PRECISION NOT NULL,
    against_flying DOUBLE PRECISION NOT NULL,
    against_psychic DOUBLE PRECISION NOT NULL,
    against_bug DOUBLE PRECISION NOT NULL,
    against_rock DOUBLE PRECISION NOT NULL,
    against_ghost DOUBLE PRECISION NOT NULL,
    against_dragon DOUBLE PRECISION NOT NULL,
    against_dark DOUBLE PRECISION NOT NULL,
    against_steel DOUBLE PRECISION NOT NULL,
    against_fairy DOUBLE PRECISION NOT NULL
)
""")

conexao.commit()

Inserindo dados no banco 

In [42]:
for _, row in df_tratado.iterrows():
    cursor.execute("""
        INSERT INTO slvr.pokemon (
            pokedex_number, pokemon_name, type_1, type_2, height, weight, hit_points,
            attack, defense, total_stats, capture_rate, generation, base_happiness,
            base_experience, exp_type, evolves_from, mega_evolution, alolan_form,
            galarian_form, forms_switchable, legendary, mythical, genderless,
            female_rate, egg_cycles, against_normal, against_fire, against_water,
            against_electric, against_grass, against_ice, against_fighting,
            against_poison, against_ground, against_flying, against_psychic,
            against_bug, against_rock, against_ghost, against_dragon, against_dark,
            against_steel, against_fairy
        ) VALUES (
            %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s
        )
    """, (
        row["pokedex_number"], row["pokemon_name"], row["type_1"], row["type_2"], row["height"], row["weight"], row["hit_points"],
        row["attack"], row["defense"], row["total_stats"], row["capture_rate"], row["generation"], row["base_happiness"],
        row["base_experience"], row["exp_type"], row["evolves_from"], row["mega_evolution"], row["alolan_form"],
        row["galarian_form"], row["forms_switchable"], row["legendary"], row["mythical"], row["genderless"],
        row["female_rate"], row["egg_cycles"], row["against_normal"], row["against_fire"], row["against_water"],
        row["against_electric"], row["against_grass"], row["against_ice"], row["against_fighting"],
        row["against_poison"], row["against_ground"], row["against_flying"], row["against_psychic"],
        row["against_bug"], row["against_rock"], row["against_ghost"], row["against_dragon"], row["against_dark"],
        row["against_steel"], row["against_fairy"]
    ))

# Commit final
conexao.commit()


Fechando a conexão com o banco!

In [43]:
conexao.close()

print("\n Load concluído!")


 Load concluído!
